# STAGNet – Protocol II: Face Alignment Network (FAN) Landmark Extraction

## Overview

| | |
|---|---|
| **Notebook purpose** | Extract facial landmark coordinates from BIWI RGB frames using the Face Alignment Network (FAN) and store the results alongside the associated head-pose labels. |
| **Landmark detector** | Face Alignment Network (FAN) |
| **Evaluation protocol** | Protocol II – subject-independent BIWI split (16 training sequences / 8 testing sequences) |
| **Expected input** | BIWI RGB frames with corresponding yaw / pitch / roll annotation files, accessed through Google Drive. |
| **Expected output** | Per-sequence landmark files containing landmark coordinates and pose labels, saved to the specified Google Drive output directory. |

> **Before running this notebook**, update `DATASET_ROOT` and `OUTPUT_ROOT` in the configuration cell below to match the paths in your Google Drive.
>
> A **GPU runtime** is recommended for FAN inference speed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------
# Configuration – update these paths before running the notebook
# ---------------------------------------------------------------
DATASET_ROOT = "/content/drive/MyDrive/STAGNet_Datasets/BIWI"
OUTPUT_ROOT  = "/content/drive/MyDrive/STAGNet_Outputs/Protocol_II"

# Protocol II sequence split
TRAIN_SEQUENCES = [
    "01", "02", "03", "04", "05", "06", "07", "08",
    "09", "10", "11", "12", "13", "14", "15", "16"
]
TEST_SEQUENCES = ["17", "18", "19", "20", "21", "22", "23", "24"]

In [ ]:
# Install dependencies
!pip install face-alignment opencv-python-headless

In [ ]:
import os
import cv2
import numpy as np
import face_alignment
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

fa = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.TWO_D,
    flip_input=False,
    device=device
)
print("FAN model loaded successfully.")

In [ ]:
def extract_landmarks_fan(image_bgr, fa_model):
    """Return 68 (x, y) landmark coordinates for the first detected face, or None."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    preds = fa_model.get_landmarks(image_rgb)
    if preds is not None and len(preds) > 0:
        return np.array(preds[0], dtype=np.float32)  # shape (68, 2)
    return None


def process_split(sequences, split_name, dataset_root, output_root, fa_model):
    """Process a list of BIWI sequences and save extracted landmarks."""
    os.makedirs(os.path.join(output_root, split_name), exist_ok=True)

    for seq in sequences:
        seq_dir = os.path.join(dataset_root, seq)
        if not os.path.isdir(seq_dir):
            print(f"[WARN] Sequence directory not found: {seq_dir}")
            continue

        landmarks_list, poses_list, frame_ids, valid_flags = [], [], [], []

        for fname in sorted(os.listdir(seq_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            img_path = os.path.join(seq_dir, fname)
            pose_path = os.path.splitext(img_path)[0] + "_pose.txt"

            if not os.path.isfile(pose_path):
                continue

            image = cv2.imread(img_path)
            if image is None:
                continue

            image_resized = cv2.resize(image, (224, 224))
            landmarks = extract_landmarks_fan(image_resized, fa_model)

            pose = np.loadtxt(pose_path)  # expected: [yaw, pitch, roll]

            valid = landmarks is not None
            landmarks_list.append(landmarks if valid
                                  else np.zeros((68, 2), dtype=np.float32))
            poses_list.append(pose)
            frame_ids.append(os.path.splitext(fname)[0])
            valid_flags.append(valid)

        out_file = os.path.join(output_root, split_name, f"seq_{seq}.npz")
        np.savez(
            out_file,
            landmarks=np.array(landmarks_list, dtype=np.float32),
            poses=np.array(poses_list, dtype=np.float32),
            frame_ids=np.array(frame_ids),
            valid=np.array(valid_flags, dtype=bool)
        )
        n_valid = sum(valid_flags)
        print(f"[{split_name}] Sequence {seq}: {len(frame_ids)} frames, "
              f"{n_valid} valid detections → {out_file}")

In [ ]:
print("Processing training split ...")
process_split(TRAIN_SEQUENCES, "train", DATASET_ROOT, OUTPUT_ROOT, fa)

print("\nProcessing testing split ...")
process_split(TEST_SEQUENCES, "test", DATASET_ROOT, OUTPUT_ROOT, fa)

print("\nDone. Landmark files saved to:", OUTPUT_ROOT)